In [3]:
import importlib
import sys
import os
import torch
import numpy as np
from tqdm.notebook import tqdm
import torch

sys.path.insert(0, '..')
sys.path.insert(1, '../../..')
sys.path.insert(0, "../../src")  # src package


## Generate Train, Val, Test

In [4]:
from perturbation_logic.activity_pertubator import (
    split_prefix_suffix_readable,
    redo_last_activity_of_prefix)
from event_log_loader_service.event_log_loader import (get_train_test_val_datasets,
                                                       extract_feature_info)
np.random.seed(17)

csv_path="../../../data/helpdesk.csv"

properties = {
    'case_name' : 'Case ID',
    'concept_name' : 'Activity',
    'timestamp_name' : 'Complete Timestamp',
    'date_format' : '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column' : 'case_elapsed_time',
    'time_since_last_event_column' : 'event_elapsed_time',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 5,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.2,
    'window_size' : 'auto',
    'categorical_columns' : ['Activity', 'Resource', 'Variant index', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup'],
    'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', ],
    'continuous_positive_columns' : [],
}


train_df, val_df, test_df,  = get_train_test_val_datasets(csv_path, properties)


print(len(train_df))

data_train = split_prefix_suffix_readable(
    train_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=2,
)

data_val = split_prefix_suffix_readable(
    val_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=2,
)

data_test = split_prefix_suffix_readable(
    test_df,
    case_column=properties["case_name"],
    activity_column=properties["concept_name"],
    min_suffix_size=2,
)

torch.save(data_train, '../../../perturbed_data/helpdesk/train.pkl')
torch.save(data_val, '../../../perturbed_data/helpdesk/val.pkl')
torch.save(data_test, '../../../perturbed_data/helpdesk/test.pkl')


#display(train_df)

#extract feature info
feature_info = extract_feature_info(val_df, properties)


28685


# Create perturbed Datasets

In [3]:
# Last Event Attack
from perturbation_logic.feature_attacks import last_event_attack

# Reset data_val_copy for feature attacks
data_val_copy = data_val.copy()

# Attacks the last event of each prefix
data_pert_last = last_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['Activity', 'Resource', 'Variant index', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup'],
    num_of_features_to_attack=3,
    magnitude=0.3,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_last, '../../../perturbed_data/helpdesk/last_event_attack.pkl')


In [4]:
# Random Event Attack
from perturbation_logic.feature_attacks import random_event_attack

# Attacks random events in each prefix with probability p
data_val_copy = data_val.copy()  
data_pert_random = random_event_attack(
    data=data_val_copy,
    properties=properties,
    feature_info=feature_info,
    attackable_features=['Activity', 'Resource', 'Variant index', 'seriousness', 'customer', 'product', 'responsible_section', 'seriousness_2', 'service_level', 'service_type', 'support_section', 'workgroup'],
    num_of_features_to_attack=2,
    event_attack_probability=0.5,
    magnitude=0.5,
    feature_range_scope='local',
    random_seed=17
)

torch.save(data_pert_random, '../../../perturbed_data/helpdesk/random_event_attack.pkl')


In [5]:
# Apply "redo last activity" augmentation to each prefix/suffix pair

data_val_copy = data_val.copy()  # Reset again
data_pert = {}
for key, (prefix_df, suffix_df) in data_val_copy.items():
    new_prefix, new_suffix = redo_last_activity_of_prefix(
        prefix_df,
        suffix_df,
        properties=properties,
    )
    data_pert[key] = (new_prefix, new_suffix)


torch.save(data_pert, '../../../perturbed_data/helpdesk/redo_pert.pkl')


In [5]:
# Apply " loop augmentation"
from perturbation_logic.structural_attacks import generate_loop_augmentation

val_loops_clean, val_loops_pert = generate_loop_augmentation(
    val_df,
    properties,
    min_suffix_size=2,
    save_path="../../../perturbed_data/helpdesk",
    save_every_n=25,
)

# Save final results
torch.save(val_loops_clean, "../../../perturbed_data/helpdesk/loop_augmentation_clean.pkl")
torch.save(val_loops_pert, "../../../perturbed_data/helpdesk/loop_augmentation_pert.pkl")
print(f"Loop augmentation: {len(val_loops_clean)} clean/pert pairs") 

Loop augmentation: 130 clean/pert pairs


In [6]:
# Summarize loop augmentation results into one combined dataset
import pickle
import os

def summarize_loop_augmentation_results(save_path, load_from_disk=False):
    """
    Merge val_loops_clean and val_loops_pert into one combined view.
    If load_from_disk=True, loads from save_path/val_loops_clean.pkl and val_loops_pert.pkl.
    Otherwise expects val_loops_clean and val_loops_pert to exist in the notebook namespace.
    Returns (val_loops_clean_merged, val_loops_pert_merged) as dicts.
    """
    if load_from_disk and save_path:
        clean_path = os.path.join(save_path, "val_loops_clean.pkl")
        pert_path = os.path.join(save_path, "val_loops_pert.pkl")
        with open(clean_path, "rb") as f:
            val_loops_clean_merged = pickle.load(f)
        with open(pert_path, "rb") as f:
            val_loops_pert_merged = pickle.load(f)
    else:
        val_loops_clean_merged = val_loops_clean.copy()
        val_loops_pert_merged = val_loops_pert.copy()
    return val_loops_clean_merged, val_loops_pert_merged

# Use in-memory results from previous cell (or set load_from_disk=True to load from save_path)
val_loops_clean_combined, val_loops_pert_combined = summarize_loop_augmentation_results(
    "../../../perturbed_data/helpdesk", load_from_disk=False
)
print(f"Combined clean: {len(val_loops_clean_combined)} entries")
print(f"Combined pert:  {len(val_loops_pert_combined)} entries")

Combined clean: 130 entries
Combined pert:  130 entries


# Compare changes

In [6]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_feature_attack_impact

# Compare clean dataset with random event attack
highlight_feature_attack_impact(
    clean_data_path='../../../perturbed_data/helpdesk/val.pkl',
    perturbed_data_path='../../../perturbed_data/helpdesk/last_event_attack.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/helpdesk/val.pkl


Loading perturbed dataset from: ../../../perturbed_data/helpdesk/last_event_attack.pkl

Clean dataset has 1898 cases
Perturbed dataset has 1898 cases

COMPARISON RESULTS

Case: Case 1, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]
    service_level:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]
    customer:
      Clean:    Value 1
      Perturbed: Value 162 [CHANGED]


Case: Case 1, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    customer:
      Clean:    Value 1
      Perturbed: Value 190 [CHANGED]


Case: Case 1, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 5 [CHANGED]
    workgroup:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]
    customer:
      Clean:    Value 1
      Perturbed: Value 44 [CHANGED]


Case: Case 1


Case: Case 1016, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 5 [CHANGED]
    product:
      Clean:    Value 3
      Perturbed: Value 9 [CHANGED]


Case: Case 1017, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    customer:
      Clean:    Value 104
      Perturbed: Value 190 [CHANGED]
    Variant index:
      Clean:    26.0
      Perturbed: 61.0 [CHANGED]


Case: Case 1017, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    Resource:
      Clean:    Value 3
      Perturbed: Value 11 [CHANGED]
    product:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]


Case: Case 1017, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    Resource:
      Clean:    Value 4
      Perturbed: Value 16 [CHANGED]
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]
    service_leve


Case: Case 1073, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    service_level:
      Clean:    Value 3
      Perturbed: Value 2 [CHANGED]
    service_type:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]
    customer:
      Clean:    Value 5
      Perturbed: Value 305 [CHANGED]


Case: Case 1074, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    seriousness_2:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]


Case: Case 1074, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    customer:
      Clean:    Value 27
      Perturbed: Value 113 [CHANGED]
    Variant index:
      Clean:    2.0
      Perturbed: 78.0 [CHANGED]


Case: Case 1074, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    seriousness_2:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]
    serv


Case: Case 111, Prefix Length: 6
  [CHANGED] Found 1 event(s) with differences

  Event 5:
    seriousness_2:
      Clean:    Value 2
      Perturbed: Value 1 [CHANGED]


Case: Case 1110, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    Resource:
      Clean:    Value 9
      Perturbed: Value 14 [CHANGED]


Case: Case 1110, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    seriousness_2:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]
    service_level:
      Clean:    Value 2
      Perturbed: Value 1 [CHANGED]
    support_section:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]


Case: Case 1110, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    Resource:
      Clean:    Value 9
      Perturbed: Value 16 [CHANGED]
    service_level:
      Clean:    Value 2
      Perturbed: Value 1 [CHANGED]
    support_section:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]


Case

...


Case: Case 1462, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    service_type:
      Clean:    Value 2
      Perturbed: Value 3 [CHANGED]
    Variant index:
      Clean:    2.0
      Perturbed: 96.0 [CHANGED]


Case: Case 1462, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]
    Variant index:
      Clean:    2.0
      Perturbed: 47.0 [CHANGED]


Case: Case 1462, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    customer:
      Clean:    Value 35
      Perturbed: Value 295 [CHANGED]
    Variant index:
      Clean:    2.0
      Perturbed: 65.0 [CHANGED]


Case: Case 1463, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    Variant index:
      Clean:    4.0
      Perturbed: 96.0 [CHANGED]


Case: Case 1463, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    respon


Case: Case 1499, Prefix Length: 9
  [CHANGED] Found 1 event(s) with differences

  Event 8:
    Variant index:
      Clean:    135.0
      Perturbed: 5.0 [CHANGED]


Case: Case 1499, Prefix Length: 10
  [CHANGED] Found 1 event(s) with differences

  Event 9:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 6 [CHANGED]
    product:
      Clean:    Value 3
      Perturbed: Value 4 [CHANGED]
    Variant index:
      Clean:    135.0
      Perturbed: 75.0 [CHANGED]


Case: Case 1499, Prefix Length: 11
  [CHANGED] Found 1 event(s) with differences

  Event 10:
    workgroup:
      Clean:    Value 3
      Perturbed: Value 2 [CHANGED]
    support_section:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]


Case: Case 1499, Prefix Length: 12
  [CHANGED] Found 1 event(s) with differences

  Event 11:
    service_level:
      Clean:    Value 2
      Perturbed: Value 3 [CHANGED]


Case: Case 15, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Even


Case: Case 1528, Prefix Length: 10
  [CHANGED] Found 1 event(s) with differences

  Event 9:
    service_level:
      Clean:    Value 2
      Perturbed: Value 3 [CHANGED]


Case: Case 1529, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    customer:
      Clean:    Value 58
      Perturbed: Value 34 [CHANGED]


Case: Case 1529, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    Resource:
      Clean:    Value 2
      Perturbed: Value 16 [CHANGED]
    product:
      Clean:    Value 2
      Perturbed: Value 13 [CHANGED]
    Variant index:
      Clean:    1.0
      Perturbed: 126.0 [CHANGED]


Case: Case 153, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    service_level:
      Clean:    Value 3
      Perturbed: Value 2 [CHANGED]
    workgroup:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]
    Variant index:
      Clean:    1.0
      Perturbed: 141.0 [CHANGED]


Case: Case 153, Prefix Leng


Case: Case 1571, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    Resource:
      Clean:    Value 14
      Perturbed: Value 9 [CHANGED]
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 5 [CHANGED]
    service_type:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]


Case: Case 1571, Prefix Length: 4
  [CHANGED] Found 1 event(s) with differences

  Event 3:
    Resource:
      Clean:    Value 14
      Perturbed: Value 12 [CHANGED]
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]
    workgroup:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]


Case: Case 1571, Prefix Length: 5
  [CHANGED] Found 1 event(s) with differences

  Event 4:
    responsible_section:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]
    Variant index:
      Clean:    142.0
      Perturbed: 125.0 [CHANGED]


Case: Case 1572, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:


Case: Case 1614, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    Resource:
      Clean:    Value 8
      Perturbed: Value 5 [CHANGED]
    workgroup:
      Clean:    Value 1
      Perturbed: Value 3 [CHANGED]
    service_type:
      Clean:    Value 1
      Perturbed: Value 2 [CHANGED]


Case: Case 1615, Prefix Length: 1
  [CHANGED] Found 1 event(s) with differences

  Event 0:
    seriousness_2:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]
    product:
      Clean:    Value 15
      Perturbed: Value 10 [CHANGED]


Case: Case 1615, Prefix Length: 2
  [CHANGED] Found 1 event(s) with differences

  Event 1:
    seriousness_2:
      Clean:    Value 1
      Perturbed: Value 4 [CHANGED]
    service_level:
      Clean:    Value 2
      Perturbed: Value 3 [CHANGED]
    customer:
      Clean:    Value 22
      Perturbed: Value 34 [CHANGED]


Case: Case 1615, Prefix Length: 3
  [CHANGED] Found 1 event(s) with differences

  Event 2:
    workgroup:
   

In [7]:
# Compare clean and perturbed datasets
from perturbation_logic.attack_impact_analyzer import highlight_structural_attack_impact

# Compare clean dataset with random event attack
highlight_structural_attack_impact(
    clean_data_path='../../../perturbed_data/helpdesk/loop_augmentation_clean.pkl',
    perturbed_data_path='../../../perturbed_data/helpdesk/loop_augmentation_pert.pkl',
    properties=properties
)

Loading clean dataset from: ../../../perturbed_data/helpdesk/loop_augmentation_clean.pkl
Loading perturbed dataset from: ../../../perturbed_data/helpdesk/loop_augmentation_pert.pkl

Clean dataset has 130 cases
Perturbed dataset has 130 cases

STRUCTURAL ATTACK COMPARISON RESULTS

Case: Case 1307, Prefix Length: 1
Prefix
Activity_seq_clean = ['Assign seriousness']
Activity_seq_pert = ['Assign seriousness', 'Assign seriousness', 'Assign seriousness']
Case_elapsed_time_clean = [0.0]
Case_elapsed_time_pert = [0.0, 4266.0, 4267.0]
Event_elapsed_time_clean = [nan]
Event_elapsed_time_pert = [nan, 4266.0, 1.0]
Day_in_week_clean = [4.0]
Day_in_week_pert = [4.0, 1.0, 2.0]
Seconds_in_day_clean = [43063.0]
Seconds_in_day_pert = [43063.0, 47850.0, 47851.0]

===
suffix
Activity_seq_clean = ['Take in charge ticket', 'Wait', 'Take in charge ticket', 'Resolve ticket', 'Closed', 'EOS', 'EOS', 'EOS', 'EOS', 'EOS']
Activity_seq_pert = ['Take in charge ticket', 'Wait', 'Take in charge ticket', 'Resolve tic